# CIFAR-10 on the **CPU** — making a 24-core Threadripper earn its keep

Same model, same engine as [`cifar10_gpu_train.ipynb`](cifar10_gpu_train.ipynb); only `device='cpu'` changes. But an *efficient* CPU run needs the opposite knobs from the GPU:

| knob | why it helps on CPU |
|---|---|
| **big batch (1024)** | small ops can't fill 24 threads; big batches give oneDNN enough work per op |
| **`channels_last` (NHWC)** | oneDNN's native conv layout — *faster* on CPU (the reverse of the GPU-fp16 result) |
| **`bf16` autocast** | this Threadripper is Zen4 with AVX-512-BF16 — a real CPU speedup |
| **few DataLoader workers** | with the model on the CPU it's the bottleneck (~850 img/s), far under what 2-4 aug workers produce |

Measured: `bf16 + channels_last + batch 1024` = **2.3x** over a naive fp32/NCHW/small-batch loop.

**On utilization:** don't expect 90%. ~50-60% *is* full — that's all 24 physical cores busy; the other 24 are SMT siblings that conv/matmul can't fill. On CPU, **throughput (img/s) is the honest metric, not %util.**

In [ ]:
# -- Shared setup: ../common has the pipeline + engine, ../a1-imagenet32 has models.py --
import os, sys
for rel in ('../common', '../a1-imagenet32'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from gpu_check import set_seed
from cifar_pipeline import load_cifar10_arrays, make_loaders
from train_engine import train
import models as M
set_seed(42)

In [ ]:
import torch
DEVICE = torch.device('cpu')
print('intra-op threads:', torch.get_num_threads(), '(physical cores)')
print('logical cores    :', os.cpu_count())

## Data + model

`make_loaders` returns the **CPU** backend for a CPU device: a torchvision `DataLoader` with a few workers, and `cfg` carrying the CPU defaults (batch 1024, `channels_last=True`, bf16). `channels_last`/`bf16` are applied to the *model* by `train_engine`.

In [ ]:
trx, tryy, tex, tey = load_cifar10_arrays()
train_iter, test_iter, cfg = make_loaders(DEVICE, trx, tryy, tex, tey)
print('backend:', cfg['backend'], '| batch', cfg['batch_size'],
      '| channels_last', cfg['channels_last'], '| amp', cfg['amp_dtype'])
model = M.build('resnet18', num_classes=10)

## Train

CPU training is ~30-50x slower than the GPU here, so this defaults to a short run for demonstration. Raise `EPOCHS` if you actually want the low-90s% (it'll take a while). Watch `img/s` — that's what the CPU knobs move.

In [ ]:
EPOCHS = 5     # CPU is slow; this is a demo. ~1-2 min/epoch on 50k images.
hist, best = train(model, train_iter, test_iter, DEVICE, epochs=EPOCHS,
                   channels_last=cfg['channels_last'], amp_dtype=cfg['amp_dtype'],
                   lr=0.1 * cfg['batch_size'] / 256, wd=5e-4)
print(f'\nBest val top-1: {best:.2%}   steady-state {sum(hist["img_s"][1:])/max(1,len(hist["img_s"])-1):,.0f} img/s')